# Data Science Tool Mastery Project
## Practical Exercises with Python's Core Data Science Libraries

**Project focus:** Demonstrate practical proficiency with NumPy, Pandas, Matplotlib, Seaborn, and Scikit-learn.

### Project objectives
1. Become familiar with industry-standard data science tools.
2. Practise data loading, inspection, cleaning, transformation, analysis, visualization, and machine learning.
3. Produce a reproducible portfolio artifact with clear explanations and results.

**Dataset:** Breast Cancer Wisconsin (Diagnostic), provided through `sklearn.datasets`. It contains 569 observations, 30 numeric measurements, and a binary diagnosis target. The dataset is used here as a learning example, not as a clinical diagnostic system.

### How to use this notebook
Run the cells from top to bottom. The dataset is loaded from Scikit-learn, so no separate data file is required. If a library is missing, install it in your environment using `pip install numpy pandas matplotlib seaborn scikit-learn`.

## 1. Import the tools

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay, roc_auc_score, RocCurveDisplay
)

RANDOM_STATE = 42
sns.set_theme(style="whitegrid", palette="deep")
pd.set_option("display.max_columns", 40)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("Seaborn:", sns.__version__)

### Tool notes
- **NumPy** supports efficient numerical arrays and calculations.
- **Pandas** provides DataFrames for structured data inspection and manipulation.
- **Matplotlib** and **Seaborn** create charts for exploratory analysis.
- **Scikit-learn** provides datasets, preprocessing, model training, validation, and evaluation.

## 2. Load data with Scikit-learn and create a Pandas DataFrame

In [ ]:
dataset = load_breast_cancer()
X = pd.DataFrame(dataset.data, columns=dataset.feature_names)
y = pd.Series(dataset.target, name="target")
df = X.copy()
df["target"] = y

target_names = dict(enumerate(dataset.target_names))
df["diagnosis"] = df["target"].map(target_names)

print("Dataset shape:", df.shape)
display(df.head())

## 3. Pandas practice: inspect and summarize the data

In [ ]:
print("Column types:")
display(df.dtypes.value_counts().rename("number_of_columns").to_frame())

print("Dataset information:")
df.info()

print("Descriptive statistics:")
display(df.describe().T.head(10))

print("Target counts:")
display(df["diagnosis"].value_counts().to_frame("count"))

**Exercise 1 – Data inspection**

Use the cells above to identify the number of observations, feature columns, data types, and target-class counts. In your own words, record whether the classes are balanced and which columns are numeric.

## 4. NumPy practice: arrays, summaries, and vectorized calculations

In [ ]:
feature_array = X.to_numpy()
print("Array dimensions:", feature_array.shape)
print("Array data type:", feature_array.dtype)

# Vectorized feature summaries
feature_means = np.mean(feature_array, axis=0)
feature_stds = np.std(feature_array, axis=0)
summary = pd.DataFrame({
    "feature": X.columns,
    "numpy_mean": feature_means,
    "numpy_std": feature_stds
})
display(summary.head(10))

# Demonstrate a boolean mask and filtering
large_mean_mask = feature_means > np.median(feature_means)
print("Features with means above the median:", int(large_mean_mask.sum()))

**Exercise 2 – NumPy practice**

Change the threshold in the boolean mask or calculate the median for a selected feature. NumPy is especially useful for fast array operations and vectorized calculations.

## 5. Data quality checks and preparation with Pandas

In [ ]:
quality_report = pd.DataFrame({
    "missing_values": df.isna().sum(),
    "unique_values": df.nunique(),
    "data_type": df.dtypes.astype(str)
})
display(quality_report.head(12))

print("Duplicate rows:", df.duplicated().sum())
print("Any missing values:", bool(df.isna().any().any()))

# Separate predictors and target
X = df[dataset.feature_names]
y = df["target"]

For this packaged dataset, the predictors are numeric and complete. In other projects, missing values, inconsistent categories, and duplicate records should be investigated before modelling. The model pipeline below scales predictors without leaking information from the test set into training.

## 6. Matplotlib practice: build a clear chart

In [ ]:
class_counts = df["diagnosis"].value_counts().reindex(dataset.target_names)
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(class_counts.index, class_counts.values)
ax.set_title("Distribution of diagnosis classes")
ax.set_xlabel("Diagnosis")
ax.set_ylabel("Number of observations")
for i, value in enumerate(class_counts.values):
    ax.text(i, value + 3, str(value), ha="center")
plt.tight_layout()
plt.show()

## 7. Seaborn practice: explore feature relationships

In [ ]:
plot_data = df[["mean radius", "mean texture", "mean perimeter", "mean area", "diagnosis"]].copy()
sns.pairplot(plot_data, hue="diagnosis", corner=True, diag_kind="hist")
plt.show()

In [ ]:
corr = X.corr(numeric_only=True)
plt.figure(figsize=(12, 9))
sns.heatmap(corr, cmap="coolwarm", center=0, linewidths=0.15)
plt.title("Correlation heatmap of numeric features")
plt.tight_layout()
plt.show()

**Exercise 3 – Visualization practice**

Compare the diagnosis groups in the pair plot. In the heatmap, look for groups of highly correlated measurements. Correlation indicates association, not causation.

## 8. Scikit-learn practice: split, scale, train, and evaluate

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

model = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=5000, random_state=RANDOM_STATE))
])

model.fit(X_train, y_train)
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print("Test accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("Test ROC-AUC:", round(roc_auc_score(y_test, y_prob), 4))
print("\nClassification report:")
print(classification_report(y_test, y_pred, target_names=dataset.target_names))

## 9. Model evaluation visualizations

In [ ]:
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=dataset.target_names)
disp.plot(cmap="Blues", values_format="d")
plt.title("Logistic regression – confusion matrix")
plt.tight_layout()
plt.show()

RocCurveDisplay.from_predictions(y_test, y_prob)
plt.title("Logistic regression – ROC curve")
plt.tight_layout()
plt.show()

## 10. Cross-validation: assess stability

In [ ]:
cv_scores = cross_val_score(model, X, y, cv=5, scoring="accuracy")
print("Five-fold accuracy scores:", np.round(cv_scores, 4))
print("Mean CV accuracy:", round(cv_scores.mean(), 4))
print("Standard deviation:", round(cv_scores.std(), 4))

## 11. Tool mastery checklist

Use this checklist when reviewing your work:

- [ ] Imported and used NumPy arrays and vectorized calculations.
- [ ] Loaded and inspected structured data with Pandas.
- [ ] Checked data quality and selected model features.
- [ ] Created a chart with Matplotlib.
- [ ] Used Seaborn for pairwise and correlation visualizations.
- [ ] Built a Scikit-learn preprocessing and modelling pipeline.
- [ ] Evaluated a model with a test set, classification metrics, confusion matrix, ROC curve, and cross-validation.
- [ ] Explained the results and limitations in plain language.

## 12. Findings, interpretation, and limitations

The workflow demonstrates how the libraries work together in a typical data science process: NumPy performs numerical operations, Pandas organizes and inspects the data, Matplotlib and Seaborn support visual exploration, and Scikit-learn handles preprocessing, modelling, and evaluation. Record the actual metric values produced by your run when writing the final report.

This is an educational classification exercise using a well-known benchmark dataset. The model output must not be interpreted as medical advice or used to make clinical decisions. A single train/test split and cross-validation provide useful practice but do not establish real-world clinical performance. Further work could include comparing additional algorithms, tuning hyperparameters, examining feature importance, and testing on an independent external dataset.

## 13. References

1. Scikit-learn documentation – Breast Cancer Wisconsin dataset: https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_breast_cancer.html
2. NumPy documentation: https://numpy.org/doc/
3. Pandas documentation: https://pandas.pydata.org/docs/
4. Matplotlib documentation: https://matplotlib.org/stable/
5. Seaborn documentation: https://seaborn.pydata.org/
6. Scikit-learn documentation: https://scikit-learn.org/stable/